In [ ]:
!pip install -q mediapipe opencv-python-headless

import cv2
import numpy as np
import mediapipe as mp
from google.colab import files
import os

# Upload a video file for testing
uploaded = files.upload()

# Initialize MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# ———————————————— Points de repère des yeux (corrigés) ————————————————
LEFT_EYE = [33, 160, 158, 133, 153, 144]  # Gauche: coin extérieur, haut, bas, coin intérieur
RIGHT_EYE = [362, 385, 387, 263, 373, 363]  # Droit: coin extérieur, haut, bas, coin intérieur

def euclidean_distance(point1, point2):
    """Distance euclidienne entre deux points"""
    return np.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

def eye_aspect_ratio(eye_points, landmarks):
    """
    Calcul de l'EAR selon la formule classique :
    EAR = (||P1-P5|| + ||P2-P4||) / (2 * ||P0-P3||)
    """
    # Points de l'œil : [P0, P1, P2, P3, P4, P5]
    p0 = landmarks[eye_points[0]]  # coin extérieur
    p1 = landmarks[eye_points[1]]  # haut 1
    p2 = landmarks[eye_points[2]]  # haut 2
    p3 = landmarks[eye_points[3]]  # coin intérieur
    p4 = landmarks[eye_points[4]]  # bas 1
    p5 = landmarks[eye_points[5]]  # bas 2

    # Calcul des distances verticales (haut-bas)
    vertical1 = euclidean_distance(p1, p5)
    vertical2 = euclidean_distance(p2, p4)
    vertical_avg = (vertical1 + vertical2) / 2

    # Calcul de la distance horizontale (coin à coin)
    horizontal = euclidean_distance(p0, p3)

    # Calcul de l'EAR
    ear = vertical_avg / horizontal
    return ear

# ———————————————— Paramètres ajustables ————————————————
EAR_THRESHOLD = 0.4  # Seuil classique
CONSECUTIVE_FRAMES = 10
# ——————————————————————————————————————————————————————————

frame_count = 0
alarm_status = False

# Get uploaded file name
video_path = list(uploaded.keys())[0]

# Open video file
cap = cv2.VideoCapture(video_path)

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Create VideoWriter for output
output_path = 'drowsiness_detection_fixed.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert frame to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb_frame)

        # Reset frame count if no face detected
        if not results.multi_face_landmarks:
            frame_count = 0
            if alarm_status:
                alarm_status = False
        else:
            # Get landmarks for first face
            face_landmarks = results.multi_face_landmarks[0]
            h, w, _ = frame.shape

            # Convert landmarks to pixel coordinates
            landmark_points = {}
            for id, landmark in enumerate(face_landmarks.landmark):
                x = int(landmark.x * w)
                y = int(landmark.y * h)
                landmark_points[id] = (x, y)

            # Calculate EAR for both eyes
            left_ear = eye_aspect_ratio(LEFT_EYE, landmark_points)
            right_ear = eye_aspect_ratio(RIGHT_EYE, landmark_points)
            avg_ear = (left_ear + right_ear) / 2

            # Debug: afficher l'EAR dans la console
            print(f"Left EAR: {left_ear:.3f}, Right EAR: {right_ear:.3f}, Avg EAR: {avg_ear:.3f}")

            # Draw landmarks on the frame
            mp_drawing.draw_landmarks(
                image=frame,
                landmark_list=face_landmarks,
                connections=mp_face_mesh.FACEMESH_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
            )

            # Check for drowsiness
            if avg_ear < EAR_THRESHOLD:
                frame_count += 1
                if frame_count >= CONSECUTIVE_FRAMES:
                    if not alarm_status:
                        alarm_status = True
                    cv2.putText(frame, "DROWSINESS DETECTED!", (10, 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            else:
                frame_count = 0
                alarm_status = False

            # Display EAR value
            cv2.putText(frame, f"EAR: {avg_ear:.2f}", (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # Write frame to output video
        out.write(frame)

    # Release resources
    cap.release()
    out.release()

# Download the output video
files.download(output_path)

Saving WIN_20251119_23_45_44_Pro.mp4 to WIN_20251119_23_45_44_Pro (2).mp4
Left EAR: 0.280, Right EAR: 0.520, Avg EAR: 0.400
Left EAR: 0.275, Right EAR: 0.532, Avg EAR: 0.404
Left EAR: 0.271, Right EAR: 0.523, Avg EAR: 0.397
Left EAR: 0.256, Right EAR: 0.522, Avg EAR: 0.389
Left EAR: 0.266, Right EAR: 0.520, Avg EAR: 0.393
Left EAR: 0.264, Right EAR: 0.530, Avg EAR: 0.397
Left EAR: 0.264, Right EAR: 0.523, Avg EAR: 0.393
Left EAR: 0.276, Right EAR: 0.529, Avg EAR: 0.403
Left EAR: 0.266, Right EAR: 0.528, Avg EAR: 0.397
Left EAR: 0.270, Right EAR: 0.534, Avg EAR: 0.402
Left EAR: 0.269, Right EAR: 0.529, Avg EAR: 0.399
Left EAR: 0.278, Right EAR: 0.532, Avg EAR: 0.405
Left EAR: 0.271, Right EAR: 0.525, Avg EAR: 0.398
Left EAR: 0.279, Right EAR: 0.549, Avg EAR: 0.414
Left EAR: 0.289, Right EAR: 0.551, Avg EAR: 0.420
Left EAR: 0.288, Right EAR: 0.544, Avg EAR: 0.416
Left EAR: 0.291, Right EAR: 0.542, Avg EAR: 0.417
Left EAR: 0.264, Right EAR: 0.547, Avg EAR: 0.405
Left EAR: 0.261, Right EAR

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>